<a href="https://colab.research.google.com/github/haidy47-design/spark_csv/blob/main/Spark_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark= SparkSession.builder\
  .appName("Spark lab2")\
  .master("local[*]")\
  .getOrCreate()

print("Spark version: ", spark.version)

Spark version:  4.0.2


In [ ]:
!wget -O sales.csv https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/by-day/2010-12-01.csv

--2026-04-13 16:06:36--  https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/by-day/2010-12-01.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 275001 (269K) [text/plain]
Saving to: ‘sales.csv’

sales.csv           100%[===================>] 268.56K  --.-KB/s    in 0.01s   

2026-04-13 16:06:37 (21.7 MB/s) - ‘sales.csv’ saved [275001/275001]



In [ ]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [ ]:
import json

customers = [
    {"CustomerID": 17850, "name": "Ahmed Ali", "age": 24, "country_group": "Middle East"},
    {"CustomerID": 13047, "name": "Sara Hassan", "age": 27, "country_group": "Europe"},
    {"CustomerID": 12583, "name": "Omar Samy", "age": 30, "country_group": "Europe"},
    {"CustomerID": 14688, "name": "Mona Adel", "age": 22, "country_group": "Europe"},
    {"CustomerID": 15311, "name": "Youssef Tarek", "age": 29, "country_group": "Europe"}
]

with open("customers.json", "w") as f:
    json.dump(customers, f)

customers_df = spark.read.json("customers.json")

In [ ]:
sales_df.printSchema()
customers_df.printSchema()
sales_df.show(5)
customers_df.show(5)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)

root
 |-- CustomerID: long (nullable = true)
 |-- age: long (nullable = true)
 |-- country_group: string (nullable = true)
 |-- name: string (nullable = true)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:

In [ ]:
sales_df.na.drop().show()
customers_df.na.drop().show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|2010-12-01 08:26:00|     7.65|   17850.0|United Kingdom|
|   536365|    21730|GLASS S

In [ ]:
from pyspark.sql.functions import col, to_timestamp,sum, count, avg,when

In [ ]:
sales_df=sales_df.withColumn("Quantity", col("Quantity").cast("int"))\
                  .withColumn("UnitPrice", col("UnitPrice").cast("double"))\
                  .withColumn("InvoiceDate", to_timestamp("InvoiceDate"))

In [ ]:
sales_df = sales_df.filter((col("Quantity") > 0) & (col("UnitPrice") > 0))

In [ ]:
sales_df = sales_df.withColumn("total_amount", col("Quantity") * col("UnitPrice"))

In [ ]:
sales_df.orderBy(col("total_amount").desc()).show(10)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|      total_amount|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|   536477|    21137|BLACK RECORD COVE...|     480|2010-12-01 12:27:00|     3.39|   16210.0|United Kingdom|            1627.2|
|   536584|   84029E|RED WOOLLY HOTTIE...|     384|2010-12-01 16:22:00|     2.95|   13777.0|United Kingdom|1132.8000000000002|
|   536387|    79321|       CHILLI LIGHTS|     192|2010-12-01 09:58:00|     3.82|   16029.0|United Kingdom| 733.4399999999999|
|   536387|    22780|LIGHT GARLAND BUT...|     192|2010-12-01 09:58:00|     3.37|   16029.0|United Kingdom|            647.04|
|   536387|    22779|WOODEN OWLS LIGHT...|     192|2010-12-01 09:58:00|     3.37|   16029.0|United Kingdom|    

In [ ]:
sales_df.groupBy("Country").agg(
    sum("total_amount").alias("total_sales"),
    count("InvoiceNo").alias("num_invoices"),
    avg("total_amount").alias("avg_order_value")
).show()

+--------------+------------------+------------+------------------+
|       Country|       total_sales|num_invoices|   avg_order_value|
+--------------+------------------+------------+------------------+
|       Germany|            261.48|          15|17.432000000000002|
|        France|            855.86|          20|            42.793|
|          EIRE| 555.3799999999999|          21|26.446666666666662|
|        Norway|1919.1400000000008|          73|  26.2895890410959|
|     Australia|            358.25|          14|25.589285714285715|
|United Kingdom| 54818.08000000008|        2927| 18.72841817560645|
|   Netherlands|192.60000000000002|           2| 96.30000000000001|
+--------------+------------------+------------+------------------+



In [ ]:
sales_df.groupBy("Description").agg(
    sum("total_amount").alias("revenue")
).orderBy(col("revenue").desc()).show(10)

+--------------------+------------------+
|         Description|           revenue|
+--------------------+------------------+
|BLACK RECORD COVE...|1830.6000000000001|
|RED WOOLLY HOTTIE...|1670.5500000000002|
|REGENCY CAKESTAND...|1434.8400000000001|
|WHITE HANGING HEA...|           1224.18|
|      DOTCOM POSTAGE|           1177.26|
|       CHILLI LIGHTS|           1109.55|
|JUMBO BAG RED RET...|            938.72|
|PAPER CHAIN KIT 5...|             891.1|
|SET OF 3 COLOURED...| 813.9000000000001|
|FAIRY TALE COTTAG...|             684.9|
+--------------------+------------------+
only showing top 10 rows


In [ ]:
sales_df.groupBy("StockCode").agg(
    sum("total_amount").alias("revenue")
).orderBy(col("revenue").desc()).show(10)

+---------+------------------+
|StockCode|           revenue|
+---------+------------------+
|    21137|1830.6000000000001|
|   84029E|1670.5500000000002|
|    22423|1434.8400000000001|
|   85123A|           1224.18|
|      DOT|           1177.26|
|    79321|           1109.55|
|   85099B|            938.72|
|    22086|             891.1|
|   35004C| 813.9000000000001|
|    22466|             684.9|
+---------+------------------+
only showing top 10 rows


In [ ]:
customers_df = customers_df.withColumn("CustomerID", col("CustomerID").cast("double"))

In [ ]:
joined_df = sales_df.join(customers_df, on="CustomerID", how="inner")

In [ ]:
joined_df.show(5)

+----------+---------+---------+--------------------+--------+-------------------+---------+--------------+------------------+---+-------------+---------+
|CustomerID|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|       Country|      total_amount|age|country_group|     name|
+----------+---------+---------+--------------------+--------+-------------------+---------+--------------+------------------+---+-------------+---------+
|   17850.0|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|United Kingdom|15.299999999999999| 24|  Middle East|Ahmed Ali|
|   17850.0|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|United Kingdom|             20.34| 24|  Middle East|Ahmed Ali|
|   17850.0|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|United Kingdom|              22.0| 24|  Middle East|Ahmed Ali|
|   17850.0|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-

In [ ]:
joined_df.groupBy("country_group").agg(
    sum("total_amount").alias("total_sales")
).show()

+-------------+------------------+
|country_group|       total_sales|
+-------------+------------------+
|       Europe|2117.4500000000003|
|  Middle East| 1499.339999999999|
+-------------+------------------+



In [ ]:
joined_df.groupBy("CustomerID", "name").agg(
    sum("total_amount").alias("total_spending")
).orderBy(col("total_spending").desc()).show(5)

+----------+-------------+------------------+
|CustomerID|         name|    total_spending|
+----------+-------------+------------------+
|   17850.0|    Ahmed Ali| 1499.339999999999|
|   12583.0|    Omar Samy|            855.86|
|   15311.0|Youssef Tarek|449.97999999999996|
|   14688.0|    Mona Adel|            444.98|
|   13047.0|  Sara Hassan| 366.6300000000001|
+----------+-------------+------------------+



In [ ]:
joined_df = joined_df.withColumn(
    "age_group",
    when(col("age") < 25, "<25")
    .when((col("age") >= 25) & (col("age") <= 34), "25-34")
    .otherwise("35+")
)

joined_df.groupBy("age_group").agg(
    avg("total_amount").alias("avg_spending")
).show()

+---------+------------------+
|age_group|      avg_spending|
+---------+------------------+
|      <25|18.876893203883476|
|    25-34|          23.22875|
+---------+------------------+



In [ ]:
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")
joined_df.createOrReplaceTempView("joined_sales")

In [ ]:
spark.sql("""
SELECT Country, SUM(total_amount) AS total_sales
FROM sales
GROUP BY Country
""").show()

+--------------+------------------+
|       Country|       total_sales|
+--------------+------------------+
|       Germany|            261.48|
|        France|            855.86|
|          EIRE| 555.3799999999999|
|        Norway|1919.1400000000008|
|     Australia|            358.25|
|United Kingdom| 54818.08000000008|
|   Netherlands|192.60000000000002|
+--------------+------------------+



In [ ]:
spark.sql("""
SELECT CustomerID, name, SUM(total_amount) AS total_spending
FROM joined_sales
GROUP BY CustomerID, name
ORDER BY total_spending DESC
LIMIT 5
""").show()

+----------+-------------+------------------+
|CustomerID|         name|    total_spending|
+----------+-------------+------------------+
|   17850.0|    Ahmed Ali| 1499.339999999999|
|   12583.0|    Omar Samy|            855.86|
|   15311.0|Youssef Tarek|449.97999999999996|
|   14688.0|    Mona Adel|            444.98|
|   13047.0|  Sara Hassan| 366.6300000000001|
+----------+-------------+------------------+



In [ ]:
spark.sql("""
SELECT age_group, AVG(total_amount) AS avg_spending
FROM joined_sales
GROUP BY age_group
""").show()

+---------+------------------+
|age_group|      avg_spending|
+---------+------------------+
|      <25|18.876893203883476|
|    25-34|          23.22875|
+---------+------------------+

